In [ ]:
import colorsys
import hashlib
import math
import os
from pathlib import Path

import h3pandas
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm
import pydeck as pdk


from lib.models.geo_model_net import ResLayer

In [ ]:
tax_2_17 = pd.read_csv(
    "/data-ssd/alex/datasets/vision-export-20240929050006-aka-2.17/taxonomy.csv"
)

In [ ]:
class TFGeoPriorModel:

    def __init__(self, model_path):
        # initialize the geo model for inference
        tf.config.set_visible_devices([], "GPU")
        visible_devices = tf.config.get_visible_devices()
        for device in visible_devices:
            assert device.device_type != "GPU"
        self.gpmodel = tf.keras.models.load_model(
            model_path,
            custom_objects={"ResLayer": ResLayer},
            compile=False
        )

    def predict(self, latitude, longitude):
        encoded_loc = TFGeoPriorModel.encode_loc([latitude], [longitude])
        return self.gpmodel(tf.convert_to_tensor(
            tf.expand_dims(encoded_loc[0], axis=0)
        ), training=False)[0]

    def features_for_one_class(self, latitude, longitude):
        """Evalutes the model for a single class and multiple locations

        Args:
            latitude (list): A list of latitudes
            longitude (list): A list of longitudes (same length as latitude)
            class_of_interest (int): The single class to eval

        Returns:
            numpy array: scores for class of interest at each location
        """

        encoded_loc = TFGeoPriorModel.encode_loc(latitude, longitude)
        loc_emb = self.gpmodel.layers[0](encoded_loc)

        # res layers - feature extraction
        x = self.gpmodel.layers[1](loc_emb)
        x = self.gpmodel.layers[2](x)
        x = self.gpmodel.layers[3](x)
        x = self.gpmodel.layers[4](x)

        return x

    def eval_one_class_from_features(self, features, class_of_interest):
        # process just the one class
        return tf.keras.activations.sigmoid(
            tf.matmul(
                tf.expand_dims(self.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
                features,
                transpose_b=True
            )
        ).numpy()

    @staticmethod
    def encode_loc(latitude, longitude):
        latitude = np.array(latitude)
        longitude = np.array(longitude)
        grid_lon = longitude.astype("float32") / 180.0
        grid_lat = latitude.astype("float32") / 90.0

        norm_loc = tf.stack([grid_lon, grid_lat], axis=1)

        encoded_loc = tf.concat([
            tf.sin(norm_loc * math.pi),
            tf.cos(norm_loc * math.pi),
        ], axis=1)
        return encoded_loc

In [29]:
gp_model_tf_sinr_coords = TFGeoPriorModel(
    "/home/alex/inatGeoModelTraining/saved_model_2_17_coords.h5"
)

In [30]:
gp_model_tf_sinr_coords.gpmodel.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 256)               1280      
                                                                 
 res_layer_20 (ResLayer)     (None, 256)               131584    
                                                                 
 res_layer_21 (ResLayer)     (None, 256)               131584    
                                                                 
 res_layer_22 (ResLayer)     (None, 256)               131584    
                                                                 
 res_layer_23 (ResLayer)     (None, 256)               131584    
                                                                 
 dense_9 (Dense)             (None, 95903)             24551168  
                                                                 
 dropout_4 (Dropout)         (None, 95903)             0

In [31]:
geo_elevation_cells = pd.read_csv(
    "/home/alex/elevation_r4_5m.csv"
).set_index("h3_04").sort_index().h3.h3_to_geo()
geo_elevation_cells["lat"] = geo_elevation_cells.geometry.y
geo_elevation_cells["lng"] = geo_elevation_cells.geometry.x
_ = geo_elevation_cells.pop("geometry")

In [32]:
geo_elevation_cells.describe()

,elevation,lat,lng
count,288122.000000,2.881220e+05,288122.000000
mean,-23005.145876,2.020244e-16,-0.371932
std,15009.330107,3.913723e+01,104.039630
min,-32768.000000,-8.995860e+01,-179.997544
25%,-32768.000000,-3.004676e+01,-90.086247
50%,-32768.000000,1.987776e-15,0.000672
75%,71.461197,3.004676e+01,89.912716
max,5705.629630,8.995860e+01,179.998888


In [33]:
geo_model_features_grid = gp_model_tf_sinr_coords.features_for_one_class(
    latitude=list(geo_elevation_cells.lat),
    longitude=list(geo_elevation_cells.lng),
)

In [34]:
TARGET_TAXON_ID = 36204

In [37]:
def h3_04_geo_results_for_taxon(taxon_id, model, geo_model_features):
    taxon = tax_2_17[tax_2_17.taxon_id==taxon_id].iloc[0]
    geo_scores = model.eval_one_class_from_features(
        geo_model_features, taxon.spatial_class_id
    )
    
    return geo_scores

In [38]:
geo_scores = h3_04_geo_results_for_taxon(
    TARGET_TAXON_ID, gp_model_tf_sinr_coords, geo_model_features_grid
)

In [40]:
geo_scores.max()

0.64248884

In [41]:
geo_score_cells_grid = geo_elevation_cells.copy()
geo_score_cells_grid["geo_score"] = tf.squeeze(geo_scores).numpy()

lower_bound_score = 0.0001

geo_score_cells_grid = geo_score_cells_grid[geo_score_cells_grid.geo_score>lower_bound_score]
geo_score_cells_grid = geo_score_cells_grid.reset_index()
len(geo_score_cells_grid)

5889

In [42]:
min_score = geo_score_cells_grid.geo_score.min()
max_score = geo_score_cells_grid.geo_score.max()
geo_score_cells_grid["scaled_geo_score"] = (
    np.log10(geo_score_cells_grid["geo_score"]) - math.log10(min_score)) / (math.log10(max_score) - math.log10(min_score)
)

def map_range(value, old_min, old_max, new_min, new_max):
    return (value - old_min) / (old_max - old_min) * (new_max - new_min) + new_min

geo_score_cells_grid["scaled_geo_score_two"] = map_range(
    np.log10(geo_score_cells_grid["geo_score"]),
    np.log10(min_score),
    np.log10(max_score),
    0.,
    1.
)

In [43]:
geo_score_cells_grid["hue"] = 211.0 / 255.0
geo_score_cells_grid["sat"] = geo_score_cells_grid["scaled_geo_score"].map(
    lambda x: map_range(x, 0.0, 1.0, 0.41, 0.9)
)
geo_score_cells_grid["val"] = geo_score_cells_grid["scaled_geo_score"].map(
    lambda x: map_range(x, 0.0, 1.0, 0.4, 0.85)
)
geo_score_cells_grid["alpha"] = geo_score_cells_grid["scaled_geo_score"]

# map_range(
#     geo_score_cells_grid["scaled_geo_score"],
#     min_score,
#     max_score,
#     0.0,
#     1.0
# )
# geo_score_cells_grid["sat"] = geo_score_cells_grid["scaled_geo_score"].apply(lambda x: map_range(x, 0.5, 1., 0.41, 0.90) if x > 0.5 else 0.41)
# geo_score_cells_grid["val"] = geo_score_cells_grid["scaled_geo_score"].apply(lambda x: map_range(x, 0.5, 1., 1., 0.85) if x > 0.5 else 1.0)
# geo_score_cells_grid["alpha"] = geo_score_cells_grid["scaled_geo_score"].apply(lambda x: 1 if x > 0.5 else map_range(x, 0., 0.5, 0., 1.))

geo_score_cells_grid["red"] = geo_score_cells_grid.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[0], axis=1)
geo_score_cells_grid["grn"] = geo_score_cells_grid.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[1], axis=1)
geo_score_cells_grid["blu"] = geo_score_cells_grid.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[2], axis=1)

In [44]:
geo_score_cells_grid

,h3_04,elevation,lat,lng,geo_score,scaled_geo_score,scaled_geo_score_two,hue,sat,val,alpha,red,grn,blu
0,840c293ffffffff,856.744186,63.721367,-150.139344,0.000158,0.052073,0.052073,0.827451,0.435516,0.423433,0.052073,0.416924,0.239021,0.423433
1,840c297ffffffff,482.500000,63.711587,-151.036295,0.000100,0.000303,0.000303,0.827451,0.410148,0.400136,0.000303,0.394344,0.236021,0.400136
2,840c299ffffffff,166.065217,64.393008,-150.034499,0.000132,0.031943,0.031943,0.827451,0.425652,0.414374,0.031943,0.408149,0.237995,0.414374
3,840c29bffffffff,491.348837,64.058875,-149.633932,0.000163,0.055584,0.055584,0.827451,0.437236,0.425013,0.055584,0.418454,0.239182,0.425013
4,840c2d1ffffffff,228.177778,65.069759,-149.923525,0.000129,0.029087,0.029087,0.827451,0.424253,0.413089,0.029087,0.406904,0.237835,0.413089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5884,84be727ffffffff,429.034483,-34.968508,148.363223,0.000113,0.013463,0.013463,0.827451,0.416597,0.406058,0.013463,0.400088,0.236896,0.406058
5885,84be72bffffffff,892.448276,-35.730204,148.221280,0.000131,0.030369,0.030369,0.827451,0.424881,0.413666,0.030369,0.407463,0.237907,0.413666
5886,84be735ffffffff,275.967742,-35.245470,147.488311,0.000147,0.043539,0.043539,0.827451,0.431334,0.419592,0.043539,0.413205,0.238608,0.419592
5887,84bee69ffffffff,389.633333,-31.333503,150.580146,0.000157,0.051708,0.051708,0.827451,0.435337,0.423268,0.051708,0.416765,0.239004,0.423268


In [45]:
view_state_grid = pdk.ViewState(
    latitude=37.7749295, 
    longitude=-122.4194155, 
    zoom=3, 
    bearing=0, 
    pitch=0
)
layer_grid = pdk.Layer(
    "H3HexagonLayer",
    geo_score_cells_grid,
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="[red * 255 * 0.3, grn * 255, blu * 255, alpha * 255]",
    get_line_color=[255, 100, 100],
    line_width_min_pixels=0,
)
r_grid = pdk.Deck(
    layers=[layer_grid], 
    initial_view_state=view_state_grid,
    map_style=pdk.map_styles.LIGHT,
)
r_grid.show()

# ok let's try with the sinr map

In [46]:
geo_elevation_cells = pd.read_csv(
    "/home/alex/elevation_r4_5m.csv"
).set_index("h3_04").sort_index().h3.h3_to_geo()
geo_elevation_cells["lat"] = geo_elevation_cells.geometry.y
geo_elevation_cells["lng"] = geo_elevation_cells.geometry.x
_ = geo_elevation_cells.pop("geometry")

In [ ]:
geo_model_features_sinr = gp_model_2_17_sinr.features_for_one_class_elevation(
    latitude=list(geo_elevation_cells.lat),
    longitude=list(geo_elevation_cells.lng),
    elevation=list(geo_elevation_cells.elevation)
)

In [ ]:
geo_scores = h3_04_geo_results_for_taxon(
    TARGET_TAXON_ID, gp_model_2_17_sinr, geo_model_features_sinr
)

In [ ]:
geo_score_cells_sinr = geo_elevation_cells.copy()
geo_score_cells_sinr["geo_score"] = tf.squeeze(geo_scores).numpy()
lower_bound_score = np.array([0.0001, sinr_threshold.loc[TARGET_TAXON_ID].thres / 10]).min()

geo_score_cells_sinr = geo_score_cells_sinr[geo_score_cells_sinr.geo_score>lower_bound_score]
geo_score_cells_sinr = geo_score_cells_sinr.reset_index()
len(geo_score_cells_sinr)

min_score = geo_score_cells_sinr.geo_score.min()
max_score = geo_score_cells_sinr.geo_score.max()
geo_score_cells_sinr["scaled_geo_score"] = (np.log10(geo_score_cells_sinr["geo_score"]) - math.log10(min_score)) / \
                    (math.log10(max_score) - math.log10(min_score))

In [ ]:
geo_score_cells_sinr["hue"] = 211.0 / 255.0
geo_score_cells_sinr["sat"] = geo_score_cells_sinr["scaled_geo_score"].map(
    lambda x: map_range(x, 0.0, 1.0, 0.41, 0.9)
)
geo_score_cells_sinr["val"] = geo_score_cells_sinr["scaled_geo_score"].map(
    lambda x: map_range(x, 0.0, 1.0, 0.4, 0.85)
)
geo_score_cells_sinr["alpha"] = geo_score_cells_sinr["scaled_geo_score"]

# geo_score_cells_sinr["hue"] = 211.0 / 255.0
# geo_score_cells_sinr["sat"] = geo_score_cells_sinr["scaled_geo_score"].apply(lambda x: map_range(x, 0., 1., 0.41, 0.90))
# geo_score_cells_sinr["val"] = geo_score_cells_sinr["scaled_geo_score"].apply(lambda x: map_range(x, 0., 1., 1., 0.85))
# geo_score_cells_sinr["alpha"] = geo_score_cells_sinr["scaled_geo_score"].apply(lambda x: 1 if x > 0.5 else map_range(x, 0., 0.5, 0., 1.))

geo_score_cells_sinr["red"] = geo_score_cells_sinr.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[0], axis=1)
geo_score_cells_sinr["grn"] = geo_score_cells_sinr.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[1], axis=1)
geo_score_cells_sinr["blu"] = geo_score_cells_sinr.apply(lambda x: colorsys.hsv_to_rgb(
    x["hue"], x["sat"], x["val"]
)[2], axis=1)

In [ ]:
view_state_sinr = pdk.ViewState(
    latitude=37.7749295, 
    longitude=-122.4194155, 
    zoom=3, 
    bearing=0, 
    pitch=0
)
layer_sinr = pdk.Layer(
    "H3HexagonLayer",
    geo_score_cells_sinr,
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="[red * 255 * 0.3, grn * 255, blu * 255, alpha * 255]",
    get_line_color=[255, 100, 100],
    line_width_min_pixels=0,
)
r_sinr = pdk.Deck(
    layers=[layer_sinr], 
    initial_view_state=view_state_sinr,
    map_style=pdk.map_styles.LIGHT,
)
r_sinr.show()